In [ ]:
from pathlib import Path

import anthropic
import pandas as pd
from dotenv import load_dotenv

In [ ]:

PATH = "/Users/none/Documents/DataAnalystAgent/data-analyst-agent"
load_dotenv(f"{PATH}/.env")
MODEL = "claude-sonnet-5-5"
PATH = Path(PATH)
df = pd.read_csv(PATH / "sales_data_sample.csv", encoding="latin-1")
client = anthropic.Anthropic()


SYSTEM = """You are a careful data analyst. A pandas DataFrame called df holds sales data.
Always call get_schema first. Write pandas code, store the answer in `result`,
and check your numbers (row counts, missing values, date ranges) before answering.
State your assumptions and give a short, clear conclusion."""


# ---------- Tools ----------

def get_schema() -> str:
    return (
        f"Columns and types:\n{df.dtypes.to_string()}\n\n"
        f"Number of rows: {len(df)}\n\n"
        f"First 5 rows:\n{df.head(5).to_string()}"
    )


def run_python(code: str) -> str:
    scope = {"df": df.copy(), "pd": pd}  # copy so the agent can't change your data
    try:
        exec(code, scope)
        return str(scope.get("result", "Code ran, but no 'result' variable was set."))[:5000]
    except Exception as e:
        return f"Python error: {e}"


TOOLS = [
    {
        "name": "get_schema",
        "description": "Show the columns, types, row count, and first rows of the sales table.",
        "input_schema": {"type": "object", "properties": {}},
    },
    {
        "name": "run_python",
        "description": "Run pandas code on the DataFrame `df`. Store the final answer in a variable named `result`.",
        "input_schema": {
            "type": "object",
            "properties": {"code": {"type": "string"}},
            "required": ["code"],
        },
    },
]

TOOL_FUNCTIONS = {
    "get_schema": lambda **_: get_schema(),
    "run_python": lambda code: run_python(code),
}


# ---------- Agent loop ----------

def analyst(question: str, max_steps: int = 10) -> str:
    messages = [{"role": "user", "content": question}]

    for _ in range(max_steps):
        response = client.messages.create(
            model=MODEL,
            max_tokens=2000,
            system=SYSTEM,
            tools=TOOLS,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            return "".join(b.text for b in response.content if b.type == "text")

        tool_results = [
            {
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": str(TOOL_FUNCTIONS[block.name](**block.input)),
            }
            for block in response.content
            if block.type == "tool_use"
        ]
        messages.append({"role": "user", "content": tool_results})

    return "Stopped: too many steps without a final answer."


if __name__ == "__main__":
    print(analyst("Which product line had the highest total sales?"))